# qang desde cero: las 15 compuertas y los 14 algoritmos en lenguaje qg

Este es el cuaderno de **inicio** de la librería `qang`. Se instala con `pip install qang` y recorre, en código, la formulación teórica del documento *Formulación qg de las compuertas y los algoritmos cuánticos principales* (`manuscript/qang_teoria_es.pdf`).

**La idea en una línea.** Un estado de $n$ qubits se describe por sus **valores qg**, $qg_P=\langle P\rangle$ para cada cadena de Pauli $P$ (para un qubit: $qg_X, qg_Y, qg_Z$, con $qg_Z=\cos\theta$), y una compuerta $U$ actúa como

$$qg'_P = qg_{U^\dagger P U}.$$

Con esa regla cada compuerta es una tabla de transformación de valores qg, y cada algoritmo una **identidad de lectura**: qué valores qg contienen la respuesta.

**Qué es y qué no es.** Es la representación de Pauli (conocida), escrita en las unidades qg que usa todo el proyecto. Aporta un lenguaje común y dos clasificaciones prácticas: qué compuertas conservan el peso de Hamming (donde funcionan el testigo y el filtro qg) y qué algoritmos se leen con pocos valores qg y cuáles necesitan el histograma completo. No es una ventaja cuántica nueva.

**Contenido**
0. Instalación
1. El marco: valores qg de un estado
2. Las 15 compuertas
3. Los 14 algoritmos
4. Resumen: qué se lee con qué, y cómo seguir

## 0. Instalación

In [1]:
# En Colab: instala qang desde PyPI (versión 0.5.0 o posterior, que trae qang.formulation).
import importlib, subprocess, sys
try:
    import qang.formulation
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qang>=0.5"], check=False)
    try:
        importlib.invalidate_caches(); import qang.formulation
    except ImportError:  # si PyPI todavía no tiene la 0.5.0, se instala desde GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/Viny2030/qang.git"], check=True)
        importlib.invalidate_caches(); import qang.formulation
import numpy as np
import qang
from qang import formulation as F
print("qang", qang.__version__)

qang 0.5.0


## 1. El marco: valores qg de un estado

`F.qg_values(estado)` devuelve todos los $qg_P$ distintos de cero; `F.state_from_qg` hace el camino inverso, $\rho=2^{-n}\sum_P qg_P\,P$. La regla de Born es $P(0)=(1+qg_Z)/2$.

In [2]:
ket0 = np.array([1, 0])
print("|0>      :", F.qg_values(ket0))
theta = 0.7
print("Ry(0.7)|0>:", {k: round(v, 4) for k, v in F.qg_values(F.ry(theta) @ ket0).items()},
      "| cos(0.7) =", round(np.cos(theta), 4), "-> la identidad qang qg_Z = cos(theta)")
print("P(0) desde qg_Z:", round(F.born_p0(np.cos(theta)), 4))
bell = F.apply_gate(F.apply_gate(F.qg_values(np.eye(4)[0]), np.kron(F.H, F.I2)), F.CX)
print("Bell (H y CNOT sobre |00>):", {k: round(v, 3) for k, v in bell.items()})
print("y de vuelta a la matriz densidad:\n", np.round(F.state_from_qg(bell, 2).real, 3))

|0>      : {'I': 1.0, 'Z': 1.0}
Ry(0.7)|0>: {'I': 1.0, 'X': 0.6442, 'Z': 0.7648} | cos(0.7) = 0.7648 -> la identidad qang qg_Z = cos(theta)
P(0) desde qg_Z: 0.8824
Bell (H y CNOT sobre |00>): {'II': 1.0, 'XX': 1.0, 'YY': -1.0, 'ZZ': 1.0}
y de vuelta a la matriz densidad:
 [[0.5 0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0.5]]


**Lectura.** Un estado de Bell no tiene ningún valor qg local (todos los $qg_Z$, $qg_X$ son 0): toda su información está en las correlaciones $qg_{XX}=1$, $qg_{YY}=-1$, $qg_{ZZ}=1$.

## 2. Las 15 compuertas

`F.gate_table(U)` da, para cada Pauli de cada qubit, la combinación $U^\dagger P U$: el valor qg nuevo de la izquierda es esa combinación de valores qg viejos. Convención: en las cadenas, la letra de la izquierda es el qubit 0 (el control de CX y CZ).

In [3]:
def show(name, U):
    tab = F.gate_table(U)
    pretty = {p: " ".join(f"{c:+.3g}·qg_{q}" for q, c in v.items()) for p, v in tab.items() if v != {p: 1.0}}
    n = int(np.log2(U.shape[0]))
    tag = "Clifford" if F.is_clifford(U) else "no Clifford"
    print(f"{name:10s} [{tag}, conserva el peso: {F.conserves_weight(U, n)}]")
    for p, s in pretty.items():
        print(f"     qg'_{p} = {s}")

t = 0.7
for name, U in [("1. X", F.X), ("2. Y", F.Y), ("3. Z", F.Z), ("4. H", F.H), ("5. S", F.S), ("6. T", F.T),
                ("7. Rx(0.7)", F.rx(t)), ("8. Ry(0.7)", F.ry(t)), ("9. Rz(0.7)", F.rz(t))]:
    show(name, U)

1. X       [Clifford, conserva el peso: False]
     qg'_Y = -1·qg_Y
     qg'_Z = -1·qg_Z
2. Y       [Clifford, conserva el peso: False]
     qg'_X = -1·qg_X
     qg'_Z = -1·qg_Z
3. Z       [Clifford, conserva el peso: True]
     qg'_X = -1·qg_X
     qg'_Y = -1·qg_Y
4. H       [Clifford, conserva el peso: False]
     qg'_X = +1·qg_Z
     qg'_Y = -1·qg_Y
     qg'_Z = +1·qg_X
5. S       [Clifford, conserva el peso: True]
     qg'_X = -1·qg_Y
     qg'_Y = +1·qg_X
6. T       [no Clifford, conserva el peso: True]
     qg'_X = +0.707·qg_X -0.707·qg_Y
     qg'_Y = +0.707·qg_X +0.707·qg_Y
7. Rx(0.7) [no Clifford, conserva el peso: False]
     qg'_Y = +0.765·qg_Y -0.644·qg_Z
     qg'_Z = +0.644·qg_Y +0.765·qg_Z
8. Ry(0.7) [no Clifford, conserva el peso: False]
     qg'_X = +0.765·qg_X +0.644·qg_Z
     qg'_Z = -0.644·qg_X +0.765·qg_Z
9. Rz(0.7) [no Clifford, conserva el peso: True]
     qg'_X = +0.765·qg_X -0.644·qg_Y
     qg'_Y = +0.644·qg_X +0.765·qg_Y
     qg'_Z = +1·qg_Z


In [4]:
for name, U in [("10. CNOT", F.CX), ("11. CZ", F.CZ), ("12. SWAP", F.SWAP), ("13. iSWAP", F.ISWAP),
                ("14. Toffoli", F.TOFFOLI), ("15. Fredkin", F.FREDKIN)]:
    show(name, U)

10. CNOT   [Clifford, conserva el peso: False]
     qg'_XI = +1·qg_XX
     qg'_YI = +1·qg_YX
     qg'_IY = +1·qg_ZY
     qg'_IZ = +1·qg_ZZ
11. CZ     [Clifford, conserva el peso: True]
     qg'_XI = +1·qg_XZ
     qg'_YI = +1·qg_YZ
     qg'_IX = +1·qg_ZX
     qg'_IY = +1·qg_ZY
12. SWAP   [Clifford, conserva el peso: True]
     qg'_XI = +1·qg_IX
     qg'_YI = +1·qg_IY
     qg'_ZI = +1·qg_IZ
     qg'_IX = +1·qg_XI
     qg'_IY = +1·qg_YI
     qg'_IZ = +1·qg_ZI
13. iSWAP  [Clifford, conserva el peso: True]
     qg'_XI = -1·qg_ZY
     qg'_YI = +1·qg_ZX
     qg'_ZI = +1·qg_IZ
     qg'_IX = -1·qg_YZ
     qg'_IY = +1·qg_XZ
     qg'_IZ = +1·qg_ZI
14. Toffoli [no Clifford, conserva el peso: False]
     qg'_XII = +0.5·qg_XII +0.5·qg_XIX +0.5·qg_XZI -0.5·qg_XZX
     qg'_YII = +0.5·qg_YII +0.5·qg_YIX +0.5·qg_YZI -0.5·qg_YZX
     qg'_IXI = +0.5·qg_IXI +0.5·qg_IXX +0.5·qg_ZXI -0.5·qg_ZXX
     qg'_IYI = +0.5·qg_IYI +0.5·qg_IYX +0.5·qg_ZYI -0.5·qg_ZYX
     qg'_IIY = +0.5·qg_IIY +0.5·qg_IZY +0.5·qg_ZIY -

15. Fredkin [no Clifford, conserva el peso: True]
     qg'_XII = +0.5·qg_XII +0.5·qg_XXX +0.5·qg_XYY +0.5·qg_XZZ
     qg'_YII = +0.5·qg_YII +0.5·qg_YXX +0.5·qg_YYY +0.5·qg_YZZ
     qg'_IXI = +0.5·qg_IIX +0.5·qg_IXI -0.5·qg_ZIX +0.5·qg_ZXI
     qg'_IYI = +0.5·qg_IIY +0.5·qg_IYI -0.5·qg_ZIY +0.5·qg_ZYI
     qg'_IZI = +0.5·qg_IIZ +0.5·qg_IZI -0.5·qg_ZIZ +0.5·qg_ZZI
     qg'_IIX = +0.5·qg_IIX +0.5·qg_IXI +0.5·qg_ZIX -0.5·qg_ZXI
     qg'_IIY = +0.5·qg_IIY +0.5·qg_IYI +0.5·qg_ZIY -0.5·qg_ZYI
     qg'_IIZ = +0.5·qg_IIZ +0.5·qg_IZI +0.5·qg_ZIZ -0.5·qg_ZZI


**Lectura.**
* **Clifford contra no Clifford.** X, Y, Z, H, S, CNOT, CZ, SWAP e iSWAP solo permutan valores qg con signo; T, las rotaciones genéricas, Toffoli y Fredkin los **mezclan** (una cadena se abre en 2 o en 4). Esa mezcla es lo que no se simula eficientemente.
* **Ceguera a la base Z.** Z, S, T, $R_z$ y CZ dejan fijos todos los $qg_Z$: su efecto solo aparece después de un Hadamard.
* **CNOT** convierte una correlación en un valor local: el $qg_Z$ del blanco pasa a ser la paridad $qg_{ZZ}$ (así se extrae un síndrome).
* **Conservación del peso.** CZ, SWAP, iSWAP y Fredkin conservan el peso de Hamming; CNOT y Toffoli no. En circuitos con compuertas que conservan el peso se aplican el testigo y el filtro qg.

In [5]:
# Toffoli comprobado sobre un estado al azar de 3 qubits: qg'_IIZ = (qg_IIZ + qg_IZZ + qg_ZIZ - qg_ZZZ)/2
rng = np.random.default_rng(1)
psi = rng.normal(size=8) + 1j * rng.normal(size=8); psi /= np.linalg.norm(psi)
q = F.qg_values(psi); q2 = F.qg_values(F.TOFFOLI @ psi)
g = lambda d, k: d.get(k, 0.0)
print("qg'_IIZ medido  :", round(g(q2, "IIZ"), 6))
print("fórmula de la tabla:", round((g(q, "IIZ") + g(q, "IZZ") + g(q, "ZIZ") - g(q, "ZZZ")) / 2, 6))

qg'_IIZ medido  : -0.183017
fórmula de la tabla: -0.183017


In [6]:
# Con Qiskit (opcional): la compuerta nativa RQangGate es Ry(arccos qg_Z) y da exactamente ese qg_Z.
try:
    from qiskit import QuantumCircuit
    from qiskit.quantum_info import Statevector
    from qang.qiskit_gate import RQangGate, SignedRQangGate
    for qz in (0.9, 0.0, -0.6):
        qc = QuantumCircuit(1); qc.append(SignedRQangGate(qz, sign=-1), [0])
        v = F.qg_values(Statevector(qc).data)
        print(f"SignedRQangGate({qz:+.1f}, s=-1): qg_Z = {v.get('Z', 0):+.3f}, qg_X = {v.get('X', 0):+.3f}")
except ImportError:
    print("Qiskit no está instalado: pip install 'qang[qiskit]'")

SignedRQangGate(+0.9, s=-1): qg_Z = +0.900, qg_X = -0.436
SignedRQangGate(+0.0, s=-1): qg_Z = +0.000, qg_X = -1.000
SignedRQangGate(-0.6, s=-1): qg_Z = -0.600, qg_X = -0.800


## 3. Los 14 algoritmos

En cada caso: la identidad de lectura en qg, comprobada sobre el estado exacto.

In [7]:
# 1. Deutsch-Jozsa y 2. Bernstein-Vazirani: alcanza con los qg_Z de cada qubit
n = 4
const = F.phase_oracle_output(np.ones(2**n))
f_bal = np.array([1, -1] * 8)
print("1. DJ  constante ->", F.deutsch_jozsa_is_constant(const, n), "| balanceada ->", F.deutsch_jozsa_is_constant(F.phase_oracle_output(f_bal), n))
s = np.array([1, 0, 1, 1])
xs = (np.arange(2**n)[:, None] >> np.arange(n)[::-1]) & 1
p_bv = F.phase_oracle_output((-1.0) ** (xs @ s))
print("2. BV  qg_Z por qubit:", [round(v, 3) for v in F.single_qg_z(p_bv, n)], "-> secreto", F.bernstein_vazirani_secret(p_bv, n))

1. DJ  constante -> True | balanceada -> False
2. BV  qg_Z por qubit: [-1.0, 1.0, -1.0, -1.0] -> secreto [1, 0, 1, 1]


In [8]:
# 3. Simon: la única paridad Z no trivial distinta de cero es el secreto
print("3. Simon, s = 101: paridades no nulas", F.simon_nonzero_parities(F.simon_output([1, 0, 1]), 3))
# 4. Shor (búsqueda del orden): con r = 4 los qg_Z locales lo muestran; con r = 6, no
for r in (4, 6):
    print(f"4. Shor, período {r}: qg_Z locales", [round(v, 2) for v in F.single_qg_z(F.order_finding_output(r, 6), 6)])

3. Simon, s = 101: paridades no nulas [(0, 0, 0), (1, 0, 1)]
4. Shor, período 4: qg_Z locales [-0.0, -0.0, 1.0, 1.0, 1.0, 1.0]
4. Shor, período 6: qg_Z locales [0.0, 0.33, 0.33, 0.31, 0.25, 0.0]


In [9]:
# 5. Grover: qg_Z de cada qubit = (-1)^bit · (N·P - 1)/(N - 1)
n, marked = 5, 19
for k in (1, 2, 4):
    p = F.grover_state(n, marked, k) ** 2
    print(f"5. Grover k={k}: P = {p[marked]:.3f}, qg_Z = {[round(v, 3) for v in F.single_qg_z(p, n)]}, "
          f"fórmula = {[round(v, 3) for v in F.grover_qg_z_prediction(n, marked, p[marked])]}, marcado leído = {F.grover_marked_from_signs(p, n)}")

5. Grover k=1: P = 0.258, qg_Z = [-0.234, 0.234, 0.234, -0.234, -0.234], fórmula = [-0.234, 0.234, 0.234, -0.234, -0.234], marcado leído = [1, 0, 0, 1, 1]
5. Grover k=2: P = 0.602, qg_Z = [-0.59, 0.59, 0.59, -0.59, -0.59], fórmula = [-0.59, 0.59, 0.59, -0.59, -0.59], marcado leído = [1, 0, 0, 1, 1]
5. Grover k=4: P = 0.999, qg_Z = [-0.999, 0.999, 0.999, -0.999, -0.999], fórmula = [-0.999, 0.999, 0.999, -0.999, -0.999], marcado leído = [1, 0, 0, 1, 1]


In [10]:
# 6. QPE y 11. retroceso de fase: la fase queda en el ángulo ecuatorial del control, qg_X + i qg_Y = e^{i phi}
phi = 0.9
qx, qy = F.kickback_control(phi)
print(f"11. retroceso: qg_X = {qx:.4f}, qg_Y = {qy:.4f}  (cos, sin = {np.cos(phi):.4f}, {np.sin(phi):.4f}) -> phi = {np.arctan2(qy, qx):.4f}")
print("6. QPE iterativa: repitiendo con U^(2^k) se lee un bit de la fase por vez, siempre desde qg_X y qg_Y del control.")

11. retroceso: qg_X = 0.6216, qg_Y = 0.7833  (cos, sin = 0.6216, 0.7833) -> phi = 0.9000
6. QPE iterativa: repitiendo con U^(2^k) se lee un bit de la fase por vez, siempre desde qg_X y qg_Y del control.


In [11]:
# 7. HHL: entrega valores qg de la solución y la probabilidad de éxito como sesgo polar de la ancilla
A = np.array([[2.0, 1.0], [1.0, 3.0]]); b = [1, 0]
qz_anc, ez = F.hhl_readout(A, b, F.Z)
x = np.linalg.solve(A, b); x = x / np.linalg.norm(x)
print(f"7. HHL: qg_Z de la ancilla = {qz_anc:.4f} (éxito {(1 - qz_anc) / 2:.3f}); qg_Z de la solución = {ez:.4f} (directo: {x[0]**2 - x[1]**2:.4f})")

7. HHL: qg_Z de la ancilla = -0.5279 (éxito 0.764); qg_Z de la solución = 0.8000 (directo: 0.8000)


In [12]:
# 8. QFT: un estado de la base se vuelve un producto de qubits ecuatoriales
n = 3
for x in (3, 5):
    psi = F.qft_matrix(n) @ np.eye(2**n)[x]
    print(f"8. QFT|{x}>:", [tuple(round(v, 3) for v in F.local_qg(psi, n, q)) for q in range(n)], " (qg_X, qg_Y, qg_Z) por qubit")

8. QFT|3>: [(-1.0, 0.0, 0.0), (-0.0, -1.0, 0.0), (-0.707, 0.707, 0.0)]  (qg_X, qg_Y, qg_Z) por qubit
8. QFT|5>: [(-1.0, 0.0, 0.0), (0.0, 1.0, 0.0), (-0.707, -0.707, 0.0)]  (qg_X, qg_Y, qg_Z) por qubit


In [13]:
# 9. VQE: la energía es lineal en los valores qg.  10. QAOA MaxCut: el costo sale de las correlaciones ZZ
H_toy = {"ZZ": 1.0, "XX": 0.5, "II": -0.2}
print("9. VQE: energía del estado de Bell =", round(F.energy_from_qg(H_toy, bell), 6), "(1 + 0.5 - 0.2)")
tri = [(0, 1, 1.0), (1, 2, 1.0), (0, 2, 1.0)]
for label in ("010", "000"):
    qg3 = F.qg_values(np.eye(8)[int(label, 2)])
    print(f"10. QAOA MaxCut en un triángulo, estado |{label}>: corte = {F.maxcut_from_qg(tri, qg3, 3)}")

9. VQE: energía del estado de Bell = 1.3 (1 + 0.5 - 0.2)
10. QAOA MaxCut en un triángulo, estado |010>: corte = 2.0
10. QAOA MaxCut en un triángulo, estado |000>: corte = 0.0


In [14]:
# 12. Conteo cuántico: una sola prueba de Hadamard, M = N (1 - qg_X) / 2
qx = F.counting_qg_x(4, [2, 7, 11])
print(f"12. conteo: qg_X = {qx:.4f} -> M = {F.count_from_qg_x(qx, 16):.3f} soluciones de 16")
# 13. Paseo cuántico de un caminante: conserva el número de excitaciones, el qg_Z medio del registro queda fijo
try:
    print("13. paseo: qg_Z medio del registro en t = 0.5 y 2.0:", [round(v, 6) for v in F.walk_register_mean(5, 4, (0.5, 2.0))], "(1 - 2/n = 0.6)")
except ImportError:
    print("13. paseo: necesita scipy")
# 14. QML: núcleo de fidelidad de estados producto desde los vectores qg locales
a = [(np.sin(0.4), 0, np.cos(0.4)), (0, 0, 1)]; b = [(np.sin(1.1), 0, np.cos(1.1)), (1, 0, 0)]
print("14. QML: núcleo producto =", round(F.product_kernel(a, b), 4), "| pureza de Bell desde qg:", round(F.purity_from_qg(F.state_from_qg(bell, 2), 2), 6))

12. conteo: qg_X = 0.6250 -> M = 3.000 soluciones de 16


13. paseo: qg_Z medio del registro en t = 0.5 y 2.0: [0.6, 0.6] (1 - 2/n = 0.6)
14. QML: núcleo producto = 0.4412 | pureza de Bell desde qg: 1.0


## 4. Resumen: qué se lee con qué

| alcanza con | algoritmos |
|---|---|
| valores locales $qg_Z$ | Deutsch–Jozsa, Bernstein–Vazirani |
| un $qg_X$, $qg_Y$ de una ancilla | retroceso de fase, QPE iterativa, conteo cuántico, éxito de HHL |
| correlaciones de pocos qubits | QAOA ($qg_{ZZ}$), VQE (cadenas del Hamiltoniano) |
| todas las paridades o el histograma | Simon, Shor con período general, Grover (la lectura local existe pero cuesta $\sim1/P^2$ disparos), núcleos de fidelidad |

**Cómo seguir con la librería**
* `qang.core.Qang` y las compuertas nativas: `qang.qiskit_gate` (`RQangGate`, `SignedRQangGate`), `qang.cirq_gate`, `qang.pennylane_gate`.
* Estimación con pocos disparos: `qang.statistics.qg_estimate(k0, n_shots, method="bayes")`.
* Filtro y testigo por sectores: `qang.sectors.filter_distribution`, `SectorExposurePass`.
* Resultados de investigación reproducibles: `notebooks/qang_avances_colab.ipynb` (§20–§74) y `notebooks/qang_full_reference.ipynb`.